# The Modern DE Stack 2026 — What I'd Actually Recommend

Complexity is the enemy of reliability.

## Recommended Stack
Kafka → Spark → Delta → dbt → Airflow → MLflow → GE → Splunk → Terraform

```
Kafka → Spark → Storage → dbt → Airflow → Serving
```

In [1]:

from confluent_kafka.admin import AdminClient
a=AdminClient({"bootstrap.servers":"localhost:9092"})
print(a.list_topics(timeout=5).topics.keys())


dict_keys(['citi.lambda.speed', 'trade_events', 'citi.decision.kafka', 'citi.alerts', 'citi.stream.e2e', 'citi.kappa.stream', '__consumer_offsets', 'citi.sysdesign.stream'])


In [2]:
import os, asyncio
# Local Spark — JRE 8 + winutils (avoids JDK-17 Netty and Windows NativeIO issues)
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
os.environ['JAVA_HOME']         = 'C:/Program Files/Java/jre1.8.0_481'
os.environ['HADOOP_HOME']       = 'C:/winutils'
os.environ['PYSPARK_PYTHON']    = 'C:/py_venv/proj_educate/Scripts/python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'C:/py_venv/proj_educate/Scripts/python.exe'
os.environ['PATH']              = 'C:/winutils/bin;' + os.environ.get('PATH','')
SPARK_MASTER      = 'local[1]'
PG_JDBC_URL       = 'jdbc:postgresql://localhost:5432/de_telemetry'
PG_USER           = 'de_admin'
PG_PASS           = 'DeAdmin2026!'
KAFKA_BOOTSTRAP   = 'localhost:9092'
DRIVER_CLASSPATH  = r'C:/Users/shareuser/.ivy2/jars/org.postgresql_postgresql-42.7.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.spark_spark-sql-kafka-0-10_2.12-3.5.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.spark_spark-token-provider-kafka-0-10_2.12-3.5.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.kafka_kafka-clients-3.4.1.jar;C:/Users/shareuser/.ivy2/jars/org.lz4_lz4-java-1.8.0.jar;C:/Users/shareuser/.ivy2/jars/org.xerial.snappy_snappy-java-1.1.10.5.jar;C:/Users/shareuser/.ivy2/jars/org.apache.commons_commons-pool2-2.11.1.jar'
print('JAVA_HOME:', os.environ['JAVA_HOME'])
print('HADOOP_HOME:', os.environ['HADOOP_HOME'])


JAVA_HOME: C:/Program Files/Java/jre1.8.0_481
HADOOP_HOME: C:/winutils


In [3]:

import subprocess
print(subprocess.run(["C:/py_venv/proj_educate/Scripts/dbt.exe","debug"],capture_output=True,text=True).stdout[:300])


00:40:46  Running with dbt=1.11.7
00:40:46  dbt version: 1.11.7
00:40:46  python version: 3.12.9
00:40:46  python path: C:\py_venv\proj_educate\Scripts\python.exe
00:40:46  os info: Windows-11-10.0.26200-SP0
00:40:46  Using profiles dir at C:\Users\shareuser\.dbt
00:40:46


In [4]:

import requests
try:
    r=requests.get("http://localhost:8082/api/v1/health",auth=("airflow","airflow"),timeout=10)
    print("Airflow health:",r.status_code,r.text[:200])
except Exception as e:
    print("Airflow not reachable:",e)


Airflow not reachable: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


In [5]:

import requests
print(requests.get("http://localhost:5000/api/2.0/mlflow/experiments/list").text[:200])


<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try agai


## What I'd Do Differently
Kafka overkill <1K events/sec  
Spark overkill <100GB  
Airflow overkill for simple jobs

## Interview Answer
Design: Kafka ingest → Spark compute → dbt transform → Airflow orchestrate → MLflow track → Splunk observe.
